# DepMap Public 25Q3：クロマチン因子パラログ合成致死スキャン（OLS/交絡補正つき）

このノートブックは DepMap Public 25Q3 のデータ（CRISPR gene effect / expression / Model metadata）を統合し、  
パラログペア（例：SMARCA4/SMARCA2）について

- **geneA が低発現（low）**のときに  
- **geneB の依存度（gene_effect）がより負（＝より必須）になるか**  

を **lineage 交絡補正（statsmodels OLS）**で一括スキャンします。

---

## 入力ファイル（同一リリース＝25Q3で揃える）
ローカルPCから Colab にアップロードしてください（後述）。

- `CRISPRGeneEffect.csv`
- `OmicsExpressionProteinCodingGenesTPMLogp1.csv`
- `Model.csv`
- `paralog_pairs.csv`（自前。2列：`geneA,geneB` のHUGO symbol）

> DepMap のリリースごとに列名（ModelID / DepMap_ID など）が揺れるため、ID列は自動検出します。


In [ ]:
#@title 0) 必要パッケージ
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests


## 1) データのアップロード
Colab 左の「Files」→「Upload」から4ファイルをアップロードし、同じディレクトリに置いてください。

（Google Drive から読み込む場合は、下のセルで drive をマウントしてパスを指定してもOK）


In [ ]:
#@title (任意) Google Drive をマウントする場合
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
#@title 2) ユーティリティ関数（ID列自動検出・low/high分割・共変量作成）
def set_index_auto(df: pd.DataFrame) -> pd.DataFrame:
    """DepMap の release で 'ModelID' / 'DepMap_ID' が揺れるので自動検出して index にする"""
    for k in ["ModelID", "DepMap_ID", "depmap_id"]:
        if k in df.columns:
            return df.set_index(k)
    # 先頭列がIDであるケースもある
    return df.set_index(df.columns[0])

def quantile_split(expr_series: pd.Series, low_q=0.2, high_q=0.8, min_n=8):
    """発現を分位点で low/high に二分（low/highの最低サンプル数 min_n）"""
    s = expr_series.dropna()
    if s.shape[0] < 2 * min_n:
        return None, None, None, None
    low_thr = s.quantile(low_q)
    high_thr = s.quantile(high_q)
    low_ids = s.index[s <= low_thr]
    high_ids = s.index[s >= high_thr]
    if (len(low_ids) < min_n) or (len(high_ids) < min_n):
        return None, None, None, None
    return low_ids, high_ids, float(low_thr), float(high_thr)

def build_covariates(model_df: pd.DataFrame, cov_col="lineage"):
    """交絡補正：カテゴリ共変量（lineage等）をダミー化"""
    if cov_col not in model_df.columns:
        raise ValueError(f"Covariate column '{cov_col}' not found. Available columns: {list(model_df.columns)[:30]}...")
    cov = pd.get_dummies(model_df[cov_col].astype("category"), drop_first=True)
    return cov


## 3) データ読み込み（25Q3）
ファイル名が異なる場合は、ここを変更してください。


In [ ]:
#@title 3) Load CSVs
GENE_EFFECT_PATH = "CRISPRGeneEffect.csv"
EXPR_PATH        = "OmicsExpressionProteinCodingGenesTPMLogp1.csv"
MODEL_PATH       = "Model.csv"
PAIR_PATH        = "paralog_pairs.csv"

gene_effect = pd.read_csv(GENE_EFFECT_PATH)
expr        = pd.read_csv(EXPR_PATH)
model       = pd.read_csv(MODEL_PATH)
pairs       = pd.read_csv(PAIR_PATH)

gene_effect = set_index_auto(gene_effect)
expr        = set_index_auto(expr)
model       = set_index_auto(model)

# 共通IDで揃える
common = gene_effect.index.intersection(expr.index).intersection(model.index)
gene_effect = gene_effect.loc[common]
expr        = expr.loc[common]
model       = model.loc[common]

print("n_models:", len(common))
print("gene_effect shape:", gene_effect.shape)
print("expr shape:", expr.shape)
print("model shape:", model.shape)
print("pairs:", pairs.shape)


## 4)（任意）コンテキスト限定（例：lungのみ）
lineage などでサブセット化できます。必要なければスキップ。


In [ ]:
#@title 4) Context filter (optional)
# 例：肺のみ
# context = model["lineage"].eq("lung")
# gene_effect = gene_effect[context]
# expr        = expr[context]
# model       = model[context]
# print("after context filter:", gene_effect.shape)


## 5) OLSでパラログ一括スキャン
- A_low：Aの発現が低い群（デフォルト：下位20%）
- A_high：Aの発現が高い群（上位20%）
- 目的：`B_effect ~ A_low + C(lineage)`
  - `beta(A_low) < 0` で「Aが低いとBがより必須」
- p値にFDR補正（BH）を適用


In [ ]:
#@title 5) Run OLS scan
# 設定
low_q  = 0.2
high_q = 0.8
min_n  = 8

cov_col = "lineage"  # Model.csv の列名に合わせて変更（例：primary_disease など）
C = build_covariates(model, cov_col=cov_col)
C_int = sm.add_constant(C, has_constant="add")

rows = []
for _, r in pairs.iterrows():
    A = r["geneA"]
    B = r["geneB"]
    if (A not in expr.columns) or (B not in gene_effect.columns):
        continue

    low_ids, high_ids, low_thr, high_thr = quantile_split(expr[A], low_q, high_q, min_n=min_n)
    if low_ids is None:
        continue

    y = gene_effect[B].copy()
    group = pd.Series(index=y.index, data=np.nan)
    group.loc[low_ids] = 1.0
    group.loc[high_ids] = 0.0

    use = group.notna() & y.notna()
    if use.sum() < (2 * min_n):
        continue

    y_use = y.loc[use].astype(float)
    g_use = group.loc[use].astype(float)

    X = C_int.loc[use].copy()
    X["A_low"] = g_use.values

    fit = sm.OLS(y_use.values, X.values).fit()

    beta = float(fit.params[X.columns.get_loc("A_low")])
    pval = float(fit.pvalues[X.columns.get_loc("A_low")])

    delta_mean = float(y.loc[low_ids].mean() - y.loc[high_ids].mean())

    rows.append({
        "geneA": A,
        "geneB": B,
        "n_low": int((use & group.eq(1)).sum()),
        "n_high": int((use & group.eq(0)).sum()),
        "low_thr": low_thr,
        "high_thr": high_thr,
        "beta_A_low(OLS)": beta,
        "p_A_low": pval,
        "delta_mean(raw)": delta_mean,
    })

res = pd.DataFrame(rows)
if res.shape[0] == 0:
    raise RuntimeError("解析できるペアがありません。gene symbol / 列名 / しきい値（min_n, quantile）を確認してください。")

res["q_A_low"] = multipletests(res["p_A_low"].values, method="fdr_bh")[1]

# ヒット（A_lowでBがより必須）
hits = (res
        .query("q_A_low < 0.05 and `beta_A_low(OLS)` < 0")
        .sort_values(["q_A_low", "beta_A_low(OLS)"]))

print("pairs scanned:", res.shape[0])
print("hits:", hits.shape[0])
hits.head(30)


## 6) 結果保存（CSV）
- 全結果：`paralog_ols_scan_results.csv`
- ヒット：`paralog_ols_scan_hits.csv`


In [ ]:
#@title 6) Save results
res.to_csv("paralog_ols_scan_results.csv", index=False)
hits.to_csv("paralog_ols_scan_hits.csv", index=False)
print("saved: paralog_ols_scan_results.csv, paralog_ols_scan_hits.csv")


## 7)（任意）特定ペアの詳細（例：SMARCA4→SMARCA2）
SMARCA4 low/high を作り、SMARCA2 dependency の差を確認します。


In [ ]:
#@title 7) Inspect one pair
A = "SMARCA4"
B = "SMARCA2"

low_ids, high_ids, low_thr, high_thr = quantile_split(expr[A], low_q, high_q, min_n=min_n)
assert low_ids is not None, "low/high分割できませんでした。min_nやquantileを調整してください。"

y = gene_effect[B]
delta = float(y.loc[low_ids].mean() - y.loc[high_ids].mean())
print(f"{A} low_thr={low_thr:.3f}, high_thr={high_thr:.3f}")
print(f"raw delta (low - high) for {B} gene_effect = {delta:.3f} (negative => more essential in A-low)")

# OLS（lineage補正）
group = pd.Series(index=y.index, data=np.nan)
group.loc[low_ids] = 1.0
group.loc[high_ids] = 0.0
use = group.notna() & y.notna()
X = C_int.loc[use].copy()
X["A_low"] = group.loc[use].astype(float).values
fit = sm.OLS(y.loc[use].astype(float).values, X.values).fit()
beta = float(fit.params[X.columns.get_loc("A_low")])
pval = float(fit.pvalues[X.columns.get_loc("A_low")])
print(f"OLS beta(A_low)={beta:.3f}, p={pval:.2e}")


---

## 補足：よくある調整ポイント
- `cov_col`：Model.csv の列名に合わせる（`lineage` が無い場合は `primary_disease` など）
- `low_q/high_q`：low/high の分位点（0.2/0.8 だと各20%）
- `min_n`：low/high群の最低サンプル数（少ないと不安定）
- lineage を限定して再実行（肺のみ等）すると交絡が減ります
